<a href="https://colab.research.google.com/github/LSRajan/valley-outfitters-analysis/blob/main/01_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/LSRajan/valley-outfitters-analysis.git
%cd valley-outfitters-analysis

import pandas as pd
pd.set_option("display.max_columns", None)

Cloning into 'valley-outfitters-analysis'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 14 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 2.21 MiB | 13.46 MiB/s, done.
/content/valley-outfitters-analysis


In [2]:
orders = pd.read_csv("data/raw/orders.csv")
customers = pd.read_csv("data/raw/customers.csv")
order_items = pd.read_csv("data/raw/order_items.csv")
products = pd.read_csv("data/raw/products.csv")
returns = pd.read_csv("data/raw/returns.csv")
web_sessions = pd.read_csv("data/raw/web_sessions.csv")

#1. Orders

In [3]:
orders.head()

,order_id,customer_id,order_date,ship_date,status,payment_method,channel,discount_code,discount_amount,shipping_cost,order_total
0,ORD-223969,C3054,2025-09-24,2025-09-26,delivered,klarna,Web,NaN,0.00,0.0,208.08
1,ORD-221807,C1857,2026-01-28,NaN,Cancelled,Gift Card,Marketplace,NaN,0.00,0.0,223.48
2,ORD-210560,C5105,2025-12-18,2025-12-24,Delivered,Card,App,NaN,0.00,0.0,109.14
3,ORD-209414,C4333,2024-07-01,2024-07-06,Delivered,Card,app,welcome10,30.17,0.0,271.56
4,ORD-219246,C5677,27.06.24,"July 01, 2024",Delivered,Apple Pay,Web,NaN,0.00,0.0,591.92


In [4]:
orders.isna().sum()

,0
order_id,0
customer_id,383
order_date,346
ship_date,3257
status,0
payment_method,0
channel,0
discount_code,19138
discount_amount,0
shipping_cost,0


###Issue 01 : Duplicate rows in orders table

In [5]:
orders.duplicated().sum()

np.int64(90)

### Proposed Fix 01: There are 90 duplicated rows in the table. Solve by dropping duplicated rows.

In [6]:
orders_temp = orders.drop_duplicates()

##1.1 order_id

In [7]:
orders["order_id"].head()

,order_id
0,ORD-223969
1,ORD-221807
2,ORD-210560
3,ORD-209414
4,ORD-219246


###Check: order_id should be unique with no duplicates

In [8]:
orders_temp["order_id"].duplicated().sum()

np.int64(0)

###Check: order_id should be in the format "ORD-[id number]"

In [9]:
pattern = r"ORD-\d+"
check = orders_temp["order_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

##1.2 customer_id

###Issue 02 : customer_id contains missing values.

In [10]:
print(orders_temp["customer_id"].isna().sum())
print(str(round(orders["customer_id"].isna().sum() / orders["customer_id"].count()*100, 2)) + "% of customer ids are missing")

382
1.55% of customer ids are missing


### Proposed Fix 02: A missing customer_id row does not make the order itself invalid, but it does break the relationship between the orders table and customers table. Any calculations involving this relationship should take this discrepancy into account, otherwise the order information can still be used without a customer id, so these rows will not be removed.


###Issue 03 : customer_id has inconsistent formatting

In [11]:
pattern = r"C\d+"
check = orders_temp["customer_id"].str.fullmatch(pattern, na = True)
check.all()

np.False_

In [12]:
orders_temp.loc[~check, "customer_id"]

,customer_id
96,c1730
293,c5806
1141,c5622
1418,c5260
1951,c2060
...,...
24282,c1545
24347,c5689
24780,c3382
24943,c5117


### Proposed Fix 03: The customer ids that do not match the format just have a lowercase prefix. To solve, capitalize all customer IDs

In [13]:
orders_temp["customer_id"] = orders_temp["customer_id"].str.upper()

/tmp/ipykernel_477/1568589473.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_temp["customer_id"] = orders_temp["customer_id"].str.upper()


In [14]:
pattern = r"C\d+"
check = orders_temp["customer_id"].str.fullmatch(pattern, na = True)
check.all()
# There are no remaining formatting issues after capitalising.

np.True_

###Issue 04 : All customer ids in the orders table should exist inside of the customers table, but many do not.

In [15]:
customers["customer_id"].head()

,customer_id
0,C3082
1,C5810
2,C6231
3,C5373
4,C3666


In [16]:
pattern = r"C\d+"
check = customers["customer_id"].str.fullmatch(pattern, na = True)
check.all()
#all customer ids in the customers table fit the pattern, so can be compared directly

np.True_

In [17]:
check = orders_temp["customer_id"].isin(customers["customer_id"])
check.all()

np.False_

In [18]:
not_present = orders_temp.loc[~check & orders_temp["customer_id"].notna(), "customer_id"]
print(not_present.count())
print(not_present.nunique())

266
236


### Proposed Fix 04: There are 266 customer ids (236 unique) that are valid in format but do not match a customer id in the customers table. To keep consistency, they can be replaced by an NA value since they do not provide any information. However, this id can still be kept to keep a record and as it may be useful if more data was provided.

##1.3 order_date

###Issue 05 : There are missing order dates

In [19]:
orders_temp["order_date"].isna().sum()

np.int64(344)

###Proposed Fix 05: Similar to before, individual missing values do not lose too much information and the row remains useable. These values/rows do not need adjustment.

###Issue 06 : Order date should have consistent formatting. Instead it has several values representing missing values, date-time instead of just date, and different representations of dates. Standard date should be YYYY-MM-DD.

In [20]:
orders_temp["order_date"].value_counts()

,count
order_date,
2026-09-30,103
0000-00-00,66
TBC,65
unknown,62
2026-09-14,44
...,...
2026-03-13 15:45:56,1
2026-08-21 11:20:12,1
2025-12-28 17:24:58,1


###(Partial) Proposed Fix 06: There are several non-missing NA values that can be replaced with NA. This would keep the values in order date as date types and keep consistent formatting for unknown order dates.

In [21]:
non_missing_na = ["TBC", "unknown", "0000-00-00"]
orders_temp["order_date"] = orders_temp["order_date"].replace(non_missing_na, pd.NA)

/tmp/ipykernel_477/2450328662.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_temp["order_date"] = orders_temp["order_date"].replace(non_missing_na, pd.NA)


In [22]:
with pd.option_context("display.max_rows", None):
    display(orders_temp["order_date"].value_counts())

,count
order_date,
2026-09-30,103
2026-09-14,44
2026-07-29,35
2025-12-25,35
2026-09-21,35
2026-09-10,35
2026-07-31,33
2025-11-09,32
2025-11-12,32


In [23]:
pattern = r"\d{4}-\d{2}-\d{2}"
check = orders_temp["order_date"].str.fullmatch(pattern, na=False)
incorrect_dates = orders_temp.loc[~check & orders_temp["order_date"].notna(), "order_date"]
incorrect_dates.value_counts()

,count
order_date,
30 Sep 2026,14
30.09.26,14
"September 30, 2026",11
09-30-2026,10
30/09/2026,9
...,...
2024-09-25 17:20:35,1
2026-02-05 15:12:27,1
31.10.24,1


###Proposed Fix 06: The remaining values are just in the wrong time formats. Get the values by their format and convert those into the correct format. Any remaining values not in a given format can be handled manually.

##1.4 ship_date

###Issue 07 : (Same issues as order date) Ship date should have consistent formatting. Instead it has several values representing missing values, date-time instead of just date, and different representations of dates. Standard date should be YYYY-MM-DD.

In [24]:
orders_temp["ship_date"].value_counts()

,count
ship_date,
TBC,52
0000-00-00,50
2025-11-09,49
2026-07-23,48
unknown,45
...,...
15.05.25,1
"March 31, 2024",1
"July 01, 2024",1


##1.5 status

###Issue 08 : While status has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value.

In [25]:
orders["status"].value_counts()

,count
status,
Delivered,14956
Cancelled,1352
completed,1039
DELIVERED,1019
Complete,998
delivered,993
Returned,991
Deliverd,987
Shipped,776


In [26]:
orders["status"].str.strip().str.lower().value_counts()

,count
status,
delivered,16968
cancelled,1594
returned,1209
completed,1039
complete,998
deliverd,987
shipped,927
pending,517
processing,391


###Proposed Fix 08: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

##1.6 payment_method

###Issue 09 : While payment method has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value. There are also different versions for abbreviated or spaced values.

In [27]:
orders["payment_method"].value_counts()

,count
payment_method,
PayPal,4056
Apple Pay,4048
Gift Card,4020
Klarna,3980
Card,3945
klarna,510
Giftcard,506
ApplePay,502
apple pay,495


###Proposed Fix 09: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

##1.7 channel

###Issue 10 : While channel has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value. There are also different versions for abbreviated or spaced values.

In [28]:
orders["channel"].value_counts()

,count
channel,
Web,11266
App,5550
Marketplace,2690
In-store,1149
web,815
WEB,806
Online,801
Phone,661
APP,381


###Proposed Fix 10: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

##1.8 discount_code

###Issue 11 : The discount code column appears to have many different variations of the same discount code. For example, 'welcome10', 'Welcome10', 'WELCOME10'. The reasonable assumption is that discount codes are not case sensitive and that these are all the same code. However it should be noted that they might actually be distinct, and that might cause an error later on.

In [29]:
orders["discount_code"].value_counts()

,count
discount_code,
WELCOME10,1718
SUMMER20,1301
FREESHIP,1025
LOYAL15,873
BF25,130
welcome10,110
Welcome10,102
WELCOME10,101
Summer20,76


###Proposed Fix 11: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying discount code. Possible downstream errors should be considered if an issue arises.

##1.9 discount_amount

###Check: discount amount should be float values to two decimal points

In [30]:
with pd.option_context("display.max_rows", None):
    display(orders_temp["discount_amount"].value_counts())

,count
discount_amount,
0.00,20278
9.76,12
1.96,11
4.55,11
4.54,10
0.72,10
18.56,10
8.30,9
20.65,9


In [31]:
orders["discount_amount"].astype(float)

,discount_amount
0,0.00
1,0.00
2,0.00
3,30.17
4,0.00
...,...
25085,0.00
25086,0.00
25087,0.00
25088,0.00


In [44]:
check = (orders["discount_amount"].round(2) != orders["discount_amount"]) & orders["discount_amount"].notna()
orders.loc[check, "discount_amount"]

,discount_amount


###Check Finding: discount amount column is already a float and checks show that no values are over 2 decimal places. Manual checks show there are no other issues.

###Check: There are no missing discount amounts

In [34]:
orders["discount_amount"].isna().sum()

np.int64(0)

##1.10 shipping_cost

###Check and Finding: Shipping costs are one of four values, each of which are valid. There are no missing values either

In [36]:
orders["shipping_cost"].value_counts(dropna=False)

,count
shipping_cost,
0.00,20949
3.99,2480
4.99,1261
6.99,400


##1.11 order_total

In [38]:
orders["order_total"].value_counts()

,count
order_total,
131.45,48
88.57,47
83.64,46
181.99,46
285.23,45
...,...
422.10,1
377.67,1
111.95,1


###Issue 12 : order_total should be float values to two decimal places. However, there are values using the prefix of "£".

In [41]:
#orders["order_total"].astype(float) generates error due to "£" sign

ValueError: could not convert string to float: '£118.81'

###Proposed Fix 12 : Replace the "£" symbol with empty string directly

In [42]:
orders_temp["order_total"] = orders_temp["order_total"].str.replace("£", "")

/tmp/ipykernel_477/2771962025.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_temp["order_total"] = orders_temp["order_total"].str.replace("£", "")


In [43]:
orders_temp["order_total"].astype(float)

,order_total
0,208.08
1,223.48
2,109.14
3,271.56
4,591.92
...,...
25084,378.52
25085,357.98
25087,465.40
25088,391.11


In [46]:
check = (orders_temp["order_total"].round(2) != orders_temp["order_total"]) & orders_temp["order_total"].notna()
orders_temp.loc[check, "order_total"]

,order_total


###Check Finding: order_total column is already a float and checks show that no values are over 2 decimal places. Manual checks show there are no other issues.